#**A6: Naive RAG vs Contextual Retrieval - st126107**

In [2]:
# ==========================================
# 0. INSTALLATIONS & SETUP
# ==========================================
!pip install -q -U langchain langchain-community langchain-groq langchain-huggingface langchain-chroma pypdf rouge-score nest_asyncio sentence-transformers pandas

import os
import json
import asyncio
import time
import nest_asyncio
import pandas as pd
from google.colab import userdata
from rouge_score import rouge_scorer
from IPython.display import display, HTML

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage
from langchain_core.documents import Document

nest_asyncio.apply()

# --- CONFIGURATION ---
STUDENT_ID = "st126107"
CHAPTER_FILE = "7.pdf"
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# MANDATORY MODEL DISCLOSURE (Task 2 Note)
RETRIEVER_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GENERATOR_MODEL = "llama-3.1-8b-instant"

llm = ChatGroq(model_name=GENERATOR_MODEL, temperature=0)
embeddings = HuggingFaceEmbeddings(model_name=RETRIEVER_MODEL)

print(f"🛠️ Retriever Model: {RETRIEVER_MODEL}")
print(f"🤖 Generator Model: {GENERATOR_MODEL}")

# ==========================================
# 1. TASK 1: DATA PREPARATION
# ==========================================
print("\n--- Task 1: Loading & Splitting Chapter 7 ---")
loader = PyPDFLoader(CHAPTER_FILE)
docs = loader.load()
full_text = " ".join([doc.page_content for doc in docs])

text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=70)
chunks = text_splitter.split_documents(docs)
print(f"✅ Extracted {len(chunks)} chunks.")

# ==========================================
# 2. TASK 2: PIPELINE IMPLEMENTATION
# ==========================================

# 2.1 Naive RAG Indexing
print("\n--- Task 2.1: Building Naive RAG ---")
naive_db = Chroma.from_documents(chunks, embeddings, collection_name=f"naive_{int(time.time())}")
naive_retriever = naive_db.as_retriever(search_kwargs={"k": 3})

# 2.2 Contextual Retrieval Indexing
async def get_contextual_chunk(content, full_doc):
    prompt = (
        f"The following is a chunk from a chapter on Large Language Models.\n"
        f"Chapter Context: {full_doc[:1000]}...\n\n"
        f"Chunk: {content}\n\n"
        f"Provide a one-sentence summary to situate this chunk within the chapter."
    )
    try:
        res = await llm.ainvoke([HumanMessage(content=prompt)])
        return f"{res.content.strip()}\n\n{content}"
    except:
        return content

async def build_contextual_docs():
    print("✨ Creating Contextual Enriched Chunks...")
    enriched_docs = []
    for i, chunk in enumerate(chunks):
        enriched_content = await get_contextual_chunk(chunk.page_content, full_text)
        enriched_docs.append(Document(page_content=enriched_content, metadata=chunk.metadata))
        if (i+1) % 25 == 0: print(f"  Processed {i+1}/{len(chunks)}...")
        await asyncio.sleep(0.4)
    return enriched_docs

contextual_chunks = asyncio.run(build_contextual_docs())
contextual_db = Chroma.from_documents(contextual_chunks, embeddings, collection_name=f"ctx_{int(time.time())}")
contextual_retriever = contextual_db.as_retriever(search_kwargs={"k": 3})

# ==========================================
# 3. TASK 2.3: EVALUATION (20 QUESTIONS)
# ==========================================
print("\n--- Task 2.3: Running Evaluation ---")

# QA Pairs defined strictly for Chapter 7
qa_pairs = [
    {"question": "What is the definition of a Large Language Model (LLM)?", "gt": "A computational agent that can interact conversationally with people, trained on massive amounts of data."},
    {"question": "Who designed ELIZA and what was its original purpose?", "gt": "Joseph Weizenbaum in 1966; it was designed to simulate a Rogerian psychologist."},
    {"question": "What are the three main stages involved in the development of a modern LLM?", "gt": "Pre-training, supervised fine-tuning (SFT), and reinforcement learning from human feedback (RLHF)."},
    {"question": "Explain 'Pre-training' in the context of LLMs.", "gt": "The stage where a model learns to predict the next word in a sequence using massive self-supervised datasets."},
    {"question": "What is 'Supervised Fine-Tuning' (SFT)?", "gt": "A process where the model is trained on curated prompt-response pairs to learn how to follow instructions."},
    {"question": "What does RLHF stand for?", "gt": "Reinforcement Learning from Human Feedback."},
    {"question": "Define 'Hallucination' as described in Chapter 7.", "gt": "When an LLM generates text that is factually incorrect while sounding confident and plausible."},
    {"question": "What are the 'HHH' criteria used in LLM alignment?", "gt": "The criteria that a model should be Helpful, Honest, and Harmless."},
    {"question": "What is the 'Turing Test' and who proposed it?", "gt": "A test of a machine's ability to exhibit behavior indistinguishable from a human, proposed by Alan Turing in 1950."},
    {"question": "What is a 'system prompt'?", "gt": "Initial instructions given to a model to define its persona, constraints, and behavior for a conversation."},
    {"question": "How does 'few-shot prompting' differ from 'zero-shot'?", "gt": "Few-shot includes examples of the task in the prompt, whereas zero-shot provides only instructions."},
    {"question": "What is 'Chain-of-Thought' (CoT) prompting?", "gt": "A technique that encourages the model to generate its step-by-step reasoning before providing a final answer."},
    {"question": "What is 'instruction tuning'?", "gt": "Fine-tuning that teaches a model to perform tasks described in natural language instructions."},
    {"question": "What is the primary risk associated with 'Stochastic Parrots'?", "gt": "The risk that the model mimics linguistic patterns without any true understanding of the underlying meaning."},
    {"question": "What is a 'Reward Model' in RLHF?", "gt": "A model trained to predict human preferences, used to score LLM outputs during the alignment process."},
    {"question": "Describe the 'Context Window' limit.", "gt": "The maximum number of tokens an LLM can process in a single prompt and response interaction."},
    {"question": "What is the Softmax temperature used for?", "gt": "A parameter that controls the randomness of the model's output; lower is more deterministic, higher is more creative."},
    {"question": "What is 'Red Teaming'?", "gt": "The practice of intentionally trying to provoke a model into generating harmful or biased content to find safety flaws."},
    {"question": "What is 'In-context learning'?", "gt": "The ability of an LLM to adapt to a new task using only examples provided in the prompt, without weight updates."},
    {"question": "Why is the evaluation of LLMs challenging?", "gt": "Because tasks are often open-ended and metrics like ROUGE may not fully capture logical correctness or helpfulness."}
]

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
results = []

for i, pair in enumerate(qa_pairs):
    q, gt = pair['question'], pair['gt']
    try:
        # Naive Answer
        n_docs = naive_retriever.invoke(q)
        n_ans = llm.invoke(f"Context: {' '.join([d.page_content for d in n_docs])}\nQ: {q}").content

        # Contextual Answer
        c_docs = contextual_retriever.invoke(q)
        c_ans = llm.invoke(f"Context: {' '.join([d.page_content for d in c_docs])}\nQ: {q}").content

        # Scoring
        n_score = scorer.score(gt, n_ans)['rougeL'].fmeasure
        c_score = scorer.score(gt, c_ans)['rougeL'].fmeasure

        results.append({
            "Question": q,
            "Naive RAG": n_ans,
            "Contextual Retrieval": c_ans,
            "Naive ROUGE-L": n_score,
            "Contextual ROUGE-L": c_score,
            "gt_hidden": gt
        })
        print(f"  [{i+1}/20] Naive: {n_score:.2f} | Contextual: {c_score:.2f}")
        time.sleep(1) # Safety cooldown
    except Exception as e:
        print(f"  ❌ Error on Q{i+1}: {e}")

# ==========================================
# 4. FINAL OUTPUT: TABLE & JSON (TASK 2 COMPLETE)
# ==========================================
df = pd.DataFrame(results)

# Formatting the table to match image_e339ed.png
display_cols = ["Question", "Naive RAG", "Contextual Retrieval", "Naive ROUGE-L", "Contextual ROUGE-L"]
styled_table = df[display_cols].style.set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#4CAF50'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('border', '1px solid black')]},
    {'selector': 'td', 'props': [('text-align', 'left'), ('border', '1px solid #ddd'), ('padding', '8px')]}
]).format({"Naive ROUGE-L": "{:.4f}", "Contextual ROUGE-L": "{:.4f}"})

print("\n" + "="*80)
print(f"📊 EVALUATION TABLE: NAIVE vs CONTEXTUAL (STUDENT ID: {STUDENT_ID})")
print("="*80)
display(styled_table)

# Summary Stats
print(f"\nAverage Naive ROUGE-L: {df['Naive ROUGE-L'].mean():.4f}")
print(f"Average Contextual ROUGE-L: {df['Contextual ROUGE-L'].mean():.4f}")

# Save JSON for Submission
os.makedirs("answer", exist_ok=True)
submission_data = [
    {
        "question": r["Question"],
        "ground_truth_answer": r["gt_hidden"],
        "naive_rag_answer": r["Naive RAG"],
        "contextual_retrieval_answer": r["Contextual Retrieval"]
    } for r in results
]

output_json = f"answer/response-{STUDENT_ID}-chapter-7.json"
with open(output_json, 'w') as f:
    json.dump(submission_data, f, indent=4)

print(f"\n✅ Submission JSON saved: {output_json}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🛠️ Retriever Model: sentence-transformers/all-MiniLM-L6-v2
🤖 Generator Model: llama-3.1-8b-instant

--- Task 1: Loading & Splitting Chapter 7 ---
✅ Extracted 155 chunks.

--- Task 2.1: Building Naive RAG ---
✨ Creating Contextual Enriched Chunks...
  Processed 25/155...
  Processed 50/155...
  Processed 75/155...
  Processed 100/155...
  Processed 125/155...
  Processed 150/155...

--- Task 2.3: Running Evaluation ---
  [1/20] Naive: 0.14 | Contextual: 0.12
  [2/20] Naive: 0.54 | Contextual: 0.50
  [3/20] Naive: 0.04 | Contextual: 0.11
  [4/20] Naive: 0.11 | Contextual: 0.11
  [5/20] Naive: 0.22 | Contextual: 0.31
  [6/20] Naive: 0.42 | Contextual: 0.17
  [7/20] Naive: 0.05 | Contextual: 0.13
  [8/20] Naive: 0.21 | Contextual: 0.06
  [9/20] Naive: 0.10 | Contextual: 0.17
  [10/20] Naive: 0.16 | Contextual: 0.07
  [11/20] Naive: 0.35 | Contextual: 0.15
  [12/20] Naive: 0.10 | Contextual: 0.09
  [13/20] Naive: 0.09 | Contextual: 0.17
  [14/20] Naive: 0.14 | Contextual: 0.15
  [15/20] Nai

,Question,Naive RAG,Contextual Retrieval,Naive ROUGE-L,Contextual ROUGE-L
0,What is the definition of a Large Language Model (LLM)?,"A Large Language Model (LLM) is a computational system that can predict the next word from previous words, given a context or prefix of words. It assigns a probability distribution over the possible next words. In essence, it's a neural network that takes as input a context or prefix and outputs a distribution over possible next words.","Based on the provided text, a Large Language Model (LLM) is a computational system that can predict the next word from previous words, given a context or prefix of words. It assigns a probability distribution over the possible next words. This is a fundamental concept that was introduced in Chapter 3, and it is being built upon in this chapter to discuss Large Language Models.",0.1370,0.1250
1,Who designed ELIZA and what was its original purpose?,"ELIZA was designed by Joseph Weizenbaum in 1966. Its original purpose was to simulate a Rogerian psychologist, and it was an experiment in computational therapy.","ELIZA was designed by Joseph Weizenbaum in 1966. Its original purpose was to simulate a Rogerian psychologist, a type of therapist who focuses on active listening and empathy.",0.5405,0.5000
2,What are the three main stages involved in the development of a modern LLM?,"Unfortunately, the provided text does not explicitly mention the three main stages involved in the development of a modern LLM. However, based on general knowledge, the development of a modern LLM typically involves the following three stages: 1. **Pretraining**: This stage involves training the LLM on a large corpus of text data to learn the patterns and relationships within the language. This is the central new idea that makes LLMs possible, as mentioned in the text. 2. **Fine-tuning**: After pretraining, the LLM is fine-tuned on a specific task or dataset to adapt its knowledge to a particular domain or application. This stage involves adjusting the LLM's parameters to optimize its performance on the target task. 3. **Deployment**: In this final stage, the fine-tuned LLM is deployed in a real-world setting, where it interacts with users and performs tasks such as answering questions, generating text, or engaging in conversation. Please note that these stages are not explicitly mentioned in the provided text, but they are generally accepted as the main stages involved in the development of a modern LLM.","The three main stages involved in the development of a modern Large Language Model (LLM) are: 1. **Pretraining**: This stage involves training the LLM on a large corpus of text data to learn general language patterns and representations. 2. **Instruction Tuning**: In this stage, the pre-trained LLM is fine-tuned on a specific task or dataset to learn how to perform a particular task, such as translation, sentiment analysis, or summarization. 3. **Preference Alignment**: This final stage involves fine-tuning the LLM to align its responses with human preferences and values, ensuring that it produces safe and responsible outputs.",0.0408,0.1062
3,Explain 'Pre-training' in the context of LLMs.,"In the context of Large Language Models (LLMs), pre-training refers to the initial training process where a model is trained on a large, general-purpose dataset, often consisting of a massive corpus of text. This training process allows the model to learn the underlying patterns, structures, and relationships within language, enabling it to develop a rich representation of language. During pre-training, the model is typically trained to predict the next word in a sequence, given the context of the previous words. This is often achieved through a self-supervised learning objective, such as masked language modeling, where some of the words in the input sequence are randomly replaced with a [MASK] token, and the model is trained to predict the original word. The pre-training process is crucial for LLMs becau


Average Naive ROUGE-L: 0.1609
Average Contextual ROUGE-L: 0.1429

✅ Submission JSON saved: answer/response-st126107-chapter-7.json
